# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [52]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
import time
import scipy.stats as st
import optuna
from sklearn.model_selection import cross_val_score

In [8]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [59]:
#dropping na, dropping irrelevant columns 
df = spaceship.dropna()
df
df = df.drop(columns = ['PassengerId','Name'])
df
columns = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
features = df[columns]

target = df['Transported']
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=42)
#MinMax Scaling & Transformation 0-1
normalizer = MinMaxScaler()
normalizer.fit(X_train)
X_train_norm = normalizer.transform(X_train)
X_test_norm = normalizer.transform(X_test)
X_train_norm, X_test_norm
X_train_norm_df = pd.DataFrame(X_train_norm, columns=X_train.columns)
X_test_norm_df = pd.DataFrame(X_train_norm, columns=X_test.columns)
X_train_norm_df
X_test_norm_df

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,0.316456,0.000000,0.056116,0.000000,0.028650,0.030094
1,0.455696,0.000000,0.088015,0.135232,0.124911,0.000049
2,0.430380,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.468354,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.278481,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
5279,0.392405,0.000000,0.000000,0.000000,0.000000,0.000000
5280,0.278481,0.003261,0.006507,0.000163,0.000000,0.018194
5281,0.316456,0.000000,0.003857,0.000000,0.081355,0.198712
5282,0.506329,0.000000,0.000704,0.000000,0.067074,0.068450


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

- Evaluate your model

In [20]:
#using KNN to train the model, checking for accuracy 
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train_norm, y_train)
y_pred = knn.predict(X_test_norm)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.7776096822995462


In [61]:
#Random Forest 
forest = RandomForestRegressor(n_estimators=100,max_depth=20)
forest.fit(X_train_norm, y_train)
RandomForestRegressor(max_depth=20)
pred = forest.predict(X_test_norm)
print("MAE", mean_absolute_error(pred, y_test))
print("RMSE", mean_squared_error(y_test, pred))
print("R2 score", forest.score(X_test_norm, y_test))

MAE 0.3072425199058661
RMSE 0.1588742191014984
R2 score 0.36441002264772426


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
#your code here
#depth of 2 
model_shallow = DecisionTreeClassifier(max_depth=2)
model_shallow.fit(X_train, y_train)

#no depth 
model_deep = DecisionTreeClassifier(max_depth=None)
model_deep.fit(X_train, y_train)

- Run Grid Search

If we are to use Random 
parameter_grid = {"max_leaf_nodes": [int(x) for x in np.linspace(start = 5, stop = 30, num = 3)],
        "max_depth":[int(x) for x in np.linspace(1, 11, num = 3)]}

In [31]:
# First we need to setup a dictionary with all the values that we want to try for each hyprerparameter

parameter_grid = {"max_depth": [10, 50],
                  "min_samples_split": [4, 16], #16 models 
                  "max_leaf_nodes": [250, 100],
                  "max_features": ["sqrt", "log2"]} # In example we're going to test 2 * 2 * 2 * 2 = 16 combinations of hyperparameters

# We create an instance or our machine learning model
dt = RandomForestRegressor(n_estimators=100,max_depth=20)

# We need to set this two variables to be able to compute a confidence interval
confidence_level = 0.95 #1 - alpha 
folds = 10 #usually 5-10 

# Now we need to create an intance of the GridSearchCV class
gs = GridSearchCV(dt, param_grid=parameter_grid, cv=folds, verbose=10) # Here the "cv" allows you to define the number of folds to use.

start_time = time.time()
gs.fit(X_train_norm_df, y_train)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")


print(f"The best combination of hyperparameters has been: {gs.best_params_}")
print(f"The R2 is: {gs.best_score_: .4f}")

results_gs_df = pd.DataFrame(gs.cv_results_).sort_values(by="mean_test_score", ascending=False)

#print(results_df.head())
gs_mean_score = results_gs_df.iloc[0,-3]
gs_sem = results_gs_df.iloc[0,-2] / np.sqrt(10)

gs_tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
gs_lower_bound = gs_mean_score - ( gs_tc * gs_sem )
gs_upper_bound = gs_mean_score + ( gs_tc * gs_sem )

print(f"The R2 confidence interval for the best combination of hyperparameters is: \
    ({gs_lower_bound: .4f}, {gs_mean_score: .4f}, {gs_upper_bound: .4f}) ")

#display(results_df)

# Let's store the best model
best_model = gs.best_estimator_

# Now is time evaluate the model in the test set
y_pred_test_df = best_model.predict(X_test_norm_df)
y_pred_test_df = best_model.predict(X_test_norm_df)

y_pred_test_df = best_model.predict(X_test_norm_df)

print("\n")
print(f"Test MAE: {mean_absolute_error(y_pred_test_df, y_test): .4f}")
print(f"Test MSE: {mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test RMSE: {root_mean_squared_error(y_pred_test_df, y_test): .4f}")
print(f"Test R2 score:  {best_model.score(X_test_norm_df, y_test): .4f}")
print("\n")



Fitting 10 folds for each of 16 candidates, totalling 160 fits
[CV 1/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 1/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.397 total time=   0.7s
[CV 2/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 2/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.448 total time=   0.7s
[CV 3/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 3/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.349 total time=   0.7s
[CV 4/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 4/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.369 total time=   0.7s
[CV 5/10; 1/16] START max_depth=10, max_features=sqrt

NameError: name 'X_test_norm_df' is not defined

- Evaluate your model

In [53]:
def objective(trial, confidence_level, folds):

    n_neighbors = trial.suggest_int("n_neighbors", 2, 25)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    p = trial.suggest_int("p", 1, 3)

    knn = KNeighborsClassifier(n_neighbors=n_neighbors,
                               weights=weights,
                               p=p)


    scorer = "accuracy" 
    

    scores = cross_val_score(knn, X_train_norm_df, y_train, cv=folds, scoring=scorer)
    
    mean_score = np.mean(scores)
    sem = np.std(scores, ddof=1) / np.sqrt(folds)

    tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
    lower_bound = mean_score - (tc * sem)
    upper_bound = mean_score + (tc * sem)

    trial.set_user_attr("CV_score_summary", [round(lower_bound, 4), round(mean_score, 4), round(upper_bound, 4)])

    return mean_score

In [54]:
confidence_level = 0.95
folds = 10

start_time = time.time()
study = optuna.create_study(direction="minimize") 
study.optimize(lambda trial: objective(trial, confidence_level, folds), n_trials=45)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")
print("The best combination of hyperparameters found was: ", study.best_params)
print(f"The best RMSE found was: {study.best_value: .4f}")

[I 2026-01-15 13:58:32,732] A new study created in memory with name: no-name-f307363d-1912-4d80-a73f-aecad30b2e24
[I 2026-01-15 13:58:34,384] Trial 0 finished with value: 0.7653222919172824 and parameters: {'n_neighbors': 25, 'weights': 'uniform', 'p': 3}. Best is trial 0 with value: 0.7653222919172824.
[I 2026-01-15 13:58:35,773] Trial 1 finished with value: 0.7710001718508335 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'p': 3}. Best is trial 0 with value: 0.7653222919172824.
[I 2026-01-15 13:58:37,159] Trial 2 finished with value: 0.7681621269404824 and parameters: {'n_neighbors': 24, 'weights': 'distance', 'p': 3}. Best is trial 0 with value: 0.7653222919172824.
[I 2026-01-15 13:58:37,591] Trial 3 finished with value: 0.7689204187431976 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.7653222919172824.
[I 2026-01-15 13:58:38,035] Trial 4 finished with value: 0.7708104198888698 and parameters: {'n_neighbors': 15, 'weights'



Time taken to find the best combination of hyperparameters among the given ones:  25.5469 seconds


The best combination of hyperparameters found was:  {'n_neighbors': 2, 'weights': 'distance', 'p': 2}
The best RMSE found was:  0.6715


As the accuracy rate already suggest, KNN also peforms better with Hyperparameter Testing. 